# CRIVO — professor do v7: falas da pessoa (A100; L4 serve)

Primeira parte do treino v7. Um modelo de terceiros (Qwen3, licença Apache 2.0, autorizado no AGENTS.md) escreve **só falas
da pessoa** para as 108 situações, em 8 estilos, e depois **confere o próprio rótulo** (escolhe, entre 5 respostas, a que
combina; fica só a fala em que ele acerta). As respostas do CRIVO nunca vêm dele. O resultado fica em
`CRIVO/dados-professor-v7/falas_professor_v7.jsonl.gz`, com modelo, revisão, data e licença em cada linha. **Não são falas
humanas.** Os pesos do professor ficam só no disco do Colab e são apagados no fim.

Este notebook instala o vLLM do mesmo jeito que a destilação (que funcionou), por isso fica **separado do treino**: o
treino (`treino_v7_entendimento_colab.ipynb`) roda depois, numa **sessão nova**, e só lê o arquivo do Drive.

> **GPU → Executar tudo.** Uns 30 a 60 min na A100. A saída do professor aparece ao vivo; se um modelo não carregar, o motivo
> aparece aqui. Se a sessão cair, rode de novo: os pedaços prontos são reaproveitados. No fim: **Ambiente de execução →
> Desconectar e excluir ambiente**, e abra o notebook do treino.

In [ ]:
# Célula 0 — GPU, Drive, código do CRIVO e vLLM (a versão que funcionou na destilação)
import os, sys, json, gzip, shutil, subprocess, hashlib, random, re, time
from pathlib import Path
from collections import Counter
os.environ["HF_HOME"] = "/content/hf"
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("ERRO: sem GPU. Ambiente de execução → Alterar tipo de ambiente → A100 (ou L4).")
print("GPU:", r.stdout.strip(), flush=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-crivo-46m" / "resumo_fase6a.json").is_file()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei CRIVO/transformer-crivo-46m. Rode na conta do treino.")
PROF_DIR = CRIVO / "dados-professor-v7"
PROF_DIR.mkdir(parents=True, exist_ok=True)
PROF_ARQ = PROF_DIR / "falas_professor_v7.jsonl.gz"
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
BRANCH = "claude/happy-mendel-wijsxp"   # depois do merge: "main"
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
EXP6 = REPO / "experimentos" / "v6_entendimento_20261019"
for p in (str(REPO), str(EXP6), str(REPO / "experimentos" / "fase5_conversa_20261013"), str(REPO / "experimentos" / "fase6b_conversa_humana")):
    if p not in sys.path:
        sys.path.insert(0, p)
if not PROF_ARQ.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.31.0"], check=True)
print("commit:", COMMIT[:10], "| saída:", PROF_ARQ, flush=True)

In [ ]:
# Célula 1 — o professor escreve falas da pessoa e confere o rótulo (pula se o arquivo já existe)
import professor_v7 as PV, detector_gatilhos as DG, gerar_v6 as G6, gerar_v6b as G6B, formas_v7
PROFESSORES = ["Qwen/Qwen3-32B-AWQ", "Qwen/Qwen3-14B-AWQ", "Qwen/Qwen3-14B", "Qwen/Qwen3-8B"]
RODAR = REPO / "experimentos" / "destilacao_20261014" / "rodar_modelo.py"
ESCOLHIDO = {}

def rodar(nome, itens):
    """Roda o primeiro professor da lista que existir, tiver licença Apache 2.0/MIT e couber. A saída aparece ao vivo."""
    entrada, saida = PROF_DIR / (nome + "_entrada.jsonl.gz"), PROF_DIR / (nome + ".jsonl.gz")
    with gzip.open(entrada, "wt", encoding="utf-8") as f:
        for x in itens:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    motivos = {}
    for modelo in ([ESCOLHIDO["m"]] if "m" in ESCOLHIDO else PROFESSORES):
        print(f"\n--- tentando {modelo}", flush=True)
        p = subprocess.Popen([sys.executable, str(RODAR), "--modelo", modelo, "--papel", "professor", "--entrada", str(entrada),
                              "--saida", str(saida)], cwd=RODAR.parent, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        ultimas = []
        for linha in p.stdout:
            ultimas = (ultimas + [linha.rstrip()])[-40:]
            if not re.search(r"it/s\]|s/it\]|Processed prompts", linha):
                print("   " + linha.rstrip(), flush=True)
        p.wait()
        if p.returncode == 0:
            ESCOLHIDO["m"] = modelo
            break
        motivos[modelo] = ultimas[-8:]
        (PROF_DIR / f"falhou_{modelo.replace('/', '_')}.txt").write_text("\n".join(ultimas))
        if p.returncode != 3:
            raise SystemExit(f"ERRO rodando {modelo} (código {p.returncode}). As últimas linhas estão acima; mande o print.")
    else:
        raise SystemExit("ERRO: nenhum professor carregou. Motivo de cada um:\n" +
                         "\n".join(f"{m}:\n   " + "\n   ".join(v) for m, v in motivos.items()))
    with gzip.open(saida, "rt", encoding="utf-8") as f:
        return {b["id"]: (b["texto"], b["fim"]) for b in map(json.loads, f)}

if not PROF_ARQ.is_file():
    S5 = G6.g1.cenas.S
    G6.revisao_v3.acrescentar_respostas(S5)
    RESP = G6.revisao_v3.respostas_para_treino(S5)
    for n in ("reciproco", "quem_sou_eu"):
        RESP[n] = list(G6.cenas_v4.S4[n]["c"])
    SITS = sorted(formas_v7.FORMAS)
    LIMPAS = {s: sorted(set(S5[s]["u"] if s in S5 else G6.cenas_v4.S4[s]["u"]) | set(formas_v7.FORMAS[s])) for s in SITS}
    TREINO_FORMAS = DG.formas_de_treino()
    CONF = PV.confundiveis(SITS, {s: TREINO_FORMAS.get(s) or LIMPAS[s] for s in SITS}, G6B.vetor)
    PED = PV.pedidos(SITS, LIMPAS, RESP)
    print(f"{len(PED)} pedidos ({len(SITS)} situações × {len(PV.ESTILOS)} estilos × {PV.FALAS_POR_PEDIDO} falas)", flush=True)
    R = rodar("v7_a_falas", [{"id": p["id"], "mensagens": p["mensagens"], "max_tokens": 900, "temperatura": 0.9} for p in PED])
    INFO = json.loads((PROF_DIR / "v7_a_falas_modelo.json").read_text())
    rr, CAND, FORA_LEITURA = random.Random(PV.SEED), [], Counter()
    for p in PED:
        falas, fora = PV.ler_falas(R.get(p["id"], ("", ""))[0])
        FORA_LEITURA.update(fora)
        CAND += [(p["situacao"], p["estilo"], u) for u in falas]
    print(f"falas lidas: {len(CAND)} | fora na leitura: {dict(FORA_LEITURA)}", flush=True)
    CONFERE = []
    for i, (s, estilo, u) in enumerate(CAND):
        msgs, letra = PV.pedido_conferencia(u, rr.choice(RESP[s]), [rr.choice(RESP[t]) for t in CONF[s]], rr)
        CONFERE.append({"id": str(i), "mensagens": msgs, "max_tokens": 4, "temperatura": 0.0, "_letra": letra})
    R2 = rodar("v7_b_conferencia", [{k: v for k, v in c.items() if k != "_letra"} for c in CONFERE])
    APROV = [CAND[int(c["id"])] for c in CONFERE if PV.ler_letra(R2.get(c["id"], ("", ""))[0]) == c["_letra"]]
    shutil.rmtree("/content/hf", ignore_errors=True)   # os pesos do professor saem do disco
    tmp = PROF_ARQ.with_name(PROF_ARQ.name + ".tmp")
    with gzip.open(tmp, "wt", encoding="utf-8") as f:
        for s, estilo, u in APROV:
            f.write(json.dumps({"situacao": s, "fala": u, "estilo": estilo, "modelo": INFO["modelo"], "revisao": INFO["revisao"],
                                "licenca": INFO["licenca"], "data": INFO["data"], "conferida_pelo_professor": True,
                                "origem": "professor (modelo de terceiros); não é fala humana"}, ensure_ascii=False) + "\n")
    tmp.rename(PROF_ARQ)
    (PROF_DIR / "manifesto_professor_v7.json").write_text(json.dumps({
        "versao": PV.VERSAO, "commit": COMMIT, "professor": INFO, "pedidos": len(PED), "falas_lidas": len(CAND),
        "fora_na_leitura": dict(FORA_LEITURA), "aprovadas_na_conferencia": len(APROV),
        "por_situacao": dict(Counter(s for s, _, _ in APROV)), "arquivo_sha256": sha256(PROF_ARQ),
        "aviso": "falas da pessoa escritas por modelo de terceiros; não são humanas (AGENTS.md)"}, ensure_ascii=False, indent=1))

In [ ]:
# Célula 2 — o que ficou: números e 40 falas sorteadas (leia: são elas que vão ensinar o CRIVO a te entender)
MP = json.loads((PROF_DIR / "manifesto_professor_v7.json").read_text())
print(f"professor: {MP['professor']['modelo']} ({MP['professor']['licenca']}) | pedidos {MP['pedidos']} | falas lidas "
      f"{MP['falas_lidas']} | aprovadas na conferência {MP['aprovadas_na_conferencia']} "
      f"({100 * MP['aprovadas_na_conferencia'] / max(1, MP['falas_lidas']):.0f}%)", flush=True)
poucas = sorted(MP["por_situacao"].items(), key=lambda x: x[1])[:10]
print("situações com menos falas aprovadas:", poucas, flush=True)
with gzip.open(PROF_ARQ, "rt", encoding="utf-8") as f:
    TODAS = [json.loads(l) for l in f]
for x in random.Random(7).sample(TODAS, min(40, len(TODAS))):
    print(f"   [{x['situacao']}] {x['fala']}", flush=True)
print("\nPronto. Agora: Ambiente de execução → Desconectar e excluir ambiente, e abra treino_v7_entendimento_colab.ipynb.", flush=True)